In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

%load_ext autoreload
%autoreload 2

sys.path.insert(0, "..")
import plasmidtools


Paths

In [ ]:
DATA_DIR = Path("../data")
FIG_DIR = DATA_DIR / "figures"
ADDGENE_DIR = DATA_DIR / "addgene"
MANUAL_DIR = DATA_DIR / "manual_annotations"

Pre-processed / pre-calculated data

Overlaps between plasmid elements and predicted CREs are calculated in the [`08_element_cre_overlap.py`](../scripts/08_element_cre_overlap.py) script.

In [ ]:
plasmid_meta = pl.read_csv(ADDGENE_DIR / "mammalian_plasmids.tsv", separator="\t")
plasmid_citations = pl.read_parquet(ADDGENE_DIR / "citations_addgene.parquet")
plasmid_stats = pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_statistics.parquet")


In [ ]:
element_positions = pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_elements.parquet")
element_citations = pl.read_parquet(ADDGENE_DIR / "citations_addgene_elements.parquet")
primer_positions = pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_primers.parquet")
primer_citations = pl.read_parquet(ADDGENE_DIR / "citations_addgene_primers.parquet")

cre_annotation = pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_cre_and_tss.parquet")

element_cre_overlap = (
    pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_element_cre_overlaps.parquet")
    .join(element_citations[["element_type", "element_name", "n_plasmids", "n_citations"]], left_on=["type", "name"], right_on=["element_type", "element_name"], how="left")
    .with_columns(pl.max_horizontal("tss_fwd_avg_signal", "tss_rev_avg_signal").alias("tss_avg_signal"))
)

primer_cre_overlap = (
    pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_primers_cre_overlaps.parquet")
    .join(primer_citations[["element_type", "element_name", "n_plasmids", "n_citations"]], left_on=["type", "name"], right_on=["element_type", "element_name"], how="left")
    .with_columns(pl.max_horizontal("tss_fwd_avg_signal", "tss_rev_avg_signal").alias("tss_avg_signal"))
)


**CRE-Element overlap statistics**

TSS Midpoints - CRE Midpoints axes

In [ ]:
df = (
    element_cre_overlap
    .filter(~pl.col("type").is_in(["enhancer", "promoter"]))
    .filter(pl.col("n_citations") >= 500)
)

fig, ax = plasmidtools.statplots.plot_regulatory_correlation(
    df,
    x_col = "n_cre_midpoints", x_label = "# CRE Midpoints per Feature Instance", x_thresh = 0.25,
    y_col = "n_tss_midpoints", y_label = "# TSS Midpoints per Feature Instance", y_thresh = 0.25,
)
# fig.savefig(FIG_DIR / "mammalian_plasmid_cre_overlap.pdf", format="pdf", bbox_inches="tight")


Fraction of CRE base pairs - CRE activity axes

In [ ]:
df = (
    element_cre_overlap
    .filter(~pl.col("type").is_in(["enhancer", "promoter"]))
    .filter(pl.col("n_citations") >= 500)
)

fig, ax = plasmidtools.statplots.plot_regulatory_correlation(
    df,
    x_col = "cre_avg_signal", x_label = "Average activity of CRE base pairs", x_thresh = 2,
    y_col = "fraction_cre_bp", y_label = "Fraction base pairs that are CRE", y_thresh = 0.1,
)


TSS Midpoints - TSS activity axes

In [ ]:
df = (
    element_cre_overlap
    .filter(~pl.col("type").is_in(["enhancer", "promoter"]))
    .filter(pl.col("n_citations") >= 500)
)

fig, ax = plasmidtools.statplots.plot_regulatory_correlation(
    df,
    x_col = "tss_avg_signal", x_label = "Average activity of TSS base pairs", x_thresh = 0.1,
    y_col = "n_tss_midpoints", y_label = "# TSS Midpoints per Feature Instance", y_thresh = 0.25,
)
plt.close()


**Functional Profiles Clustering**

Candidate cryptic CREs list is inferred in the [`10_element_cccs.py`](../scripts/10_element_cccs.py) script.

In [ ]:
df_clustered = pl.read_csv(ADDGENE_DIR / "element_cre_overlap_clustering.csv")
df_heatmap = pl.read_csv(ADDGENE_DIR / "element_cre_overlap_clustering_heatmap.csv")

N_CITATIONS_FILTER = 50
CRE_FILTER = df_clustered["is_cryptic_cre"] & ((df_clustered["n_citations"] >= N_CITATIONS_FILTER) | df_clustered["n_citations"].is_null())
CLUST1_FILTER = df_clustered["priority_group"] == 1
cryptic_cre_candidates = df_clustered.filter(CRE_FILTER)

# clust_df, heat_df = df_clustered, df_heatmap  # ALL
# clust_df, heat_df = df_clustered.filter(CLUST1_FILTER), df_heatmap.filter(CLUST1_FILTER)  # Top-ranked CREs (including known promoters)
clust_df, heat_df = df_clustered.filter(CRE_FILTER), df_heatmap.filter(CRE_FILTER)  # Highly cited Cryptic CREs

cg = plasmidtools.statplots.functional_profiling_plot(clust_df, heat_df)
ax_heat = cg.ax_heatmap
ax_heat.set_ylabel(f"Length ≥ 100 (N = {len(clust_df)})", fontsize=12, fontweight='bold')
ax_heat.set_title("Candidate Cryptic CREs", fontsize=25)


**CRE-Primer (buffer sequence) overlap statistics**

In [ ]:
fig, ax = plasmidtools.statplots.plot_regulatory_correlation(
    primer_cre_overlap,
    x_col = "n_cre_midpoints", x_label = "# CRE Midpoints per Feature Instance", x_thresh=0.25,
    y_col = "n_tss_midpoints", y_label = "# TSS Midpoints per Feature Instance", y_thresh=0.25,
)
plt.close()


**Pile-ups**

In [ ]:
PILEUP_PATH = ADDGENE_DIR / "plasmid_elements_prediction_pileups.h5"

element_type = "repeat_region"
element_name = "ITR"
(
    element_size, flank_size,
    type_matrix, cre_matrix, tss_fwd_matrix, tss_rev_matrix,
    pred_matrix, pred_fwd_matrix, pred_rev_matrix
) = plasmidtools.helpers.load_aligned_predictions_h5(element_type, element_name, PILEUP_PATH)



Element-aligned activity track pile ups

In [ ]:
fig, (ax1, ax2) = plasmidtools.statplots.combined_prediction_pileups(pred_matrix, pred_fwd_matrix, pred_rev_matrix, element_type, element_name, flank_size)


**Plasmid Context**

In [ ]:
fig, ax = plasmidtools.statplots.element_centered_architecture_heatmap(
    type_matrix=type_matrix[:10000],
    flank_size=flank_size,
    element_size=element_size,
    cre_matrix=cre_matrix,
    cre_label="CREST (HEK293T)",
    target_label=f"{element_type} - {element_name}"
)


**Representative sequences**

Sequence representation

In [ ]:
repr_seqs_dct = plasmidtools.helpers.load_representative_sequences(ADDGENE_DIR / "element_representative_sequences.fasta")
repr_seqs_cre = pl.read_parquet(ADDGENE_DIR / "element_representative_sequences_cre_overlaps.parquet")


In [ ]:
repr_seqs_dct["rep_origin", "ori"]

In [ ]:
repr_meta = []
for (type, name) in cryptic_cre_candidates[["type", "name"]].rows():
    el_dct = repr_seqs_dct[(type, name)]
    repr_meta.append({
        "element_type": type,
        "element_name": name,
        "n_plasmids": el_dct["n_plasmids"],
        "n_citations": el_dct["n_citations"],
        "citation_frequency": el_dct["citation_frequency"],
        "plasmid_frequency": el_dct["plasmid_frequency"],
    })
repr_meta = pl.DataFrame(repr_meta)

repr_meta.sort("citation_frequency")

**Contribution Scores**

In [ ]:
CONTRIBS_OUTPUT = ADDGENE_DIR / "CCCs_contrib_scores.h5"
PUFFIN_FLANK = 325

element_type, element_name = "misc_feature", "Rosa26 left arm"

element_seq_dct = repr_seqs_dct[(element_type, element_name)]
cre_positions = repr_seqs_cre.filter((pl.col("element_type") == element_type) & (pl.col('element_name') == element_name))["CREST (HEK293T)"][0].to_list()
tss_fwd_positions = repr_seqs_cre.filter((pl.col("element_type") == element_type) & (pl.col('element_name') == element_name))["Puffin (FANTOM_CAGE_fwd)"][0].to_list()
tss_rev_positions = repr_seqs_cre.filter((pl.col("element_type") == element_type) & (pl.col('element_name') == element_name))["Puffin (FANTOM_CAGE_rev)"][0].to_list()

contrib_scores = plasmidtools.helpers.load_contribution_scores(CONTRIBS_OUTPUT, element_type, element_name)

# print(contrib_scores['onehot'].shape)  # (4, L)
# print(contrib_scores['crest'].shape)  # (4, L); should be dot-producted with onehots
# print(contrib_scores['puffin_fwd'].shape)  # (L - 2 * PUFFIN_FLANK,); should be dot-producted with onehots
# print(contrib_scores['puffin_rev'].shape)  # (L - 2 * PUFFIN_FLANK,); should be dot-producted with onehots

cre_contribs = contrib_scores['onehot'] * contrib_scores['crest']
fwd_contribs = contrib_scores['onehot'][:, PUFFIN_FLANK: -PUFFIN_FLANK] * contrib_scores['puffin_fwd'][None, :]
rev_contribs = contrib_scores['onehot'][:, PUFFIN_FLANK: -PUFFIN_FLANK] * contrib_scores['puffin_rev'][None, :]

flank_size = element_seq_dct["flank_size"]
element_size = cre_contribs.shape[1] - 2 * flank_size


In [ ]:
if cre_positions:
    cre_start, cre_stop = cre_positions[0]  # TODO: iterate over all CREs
    mid = (cre_start + cre_stop) // 2
    start, stop = max(0, mid - 150), min(cre_contribs.shape[1], mid + 150)
    fig, ax = plasmidtools.contribplots.contribution_scores_plot(cre_contribs[:, start: stop])
    fig, ax = plasmidtools.contribplots.apply_element_annotations(fig, ax, slice_start=start, slice_end=stop, flank_size=flank_size, element_size=element_size, element_label=f"{element_type}-{element_name}")
    ax.set_ylabel("CREST (HEK293T)", fontsize=25)
    plt.show()
    plt.close()

if tss_fwd_positions:
    tss_start, tss_stop = tss_fwd_positions[0]  # TODO: iterate over all CREs
    mid = (tss_start + tss_stop) // 2
    start, stop = max(0, mid - 150), min(fwd_contribs.shape[1], mid + 150)
    fig, ax = plasmidtools.contribplots.contribution_scores_plot(fwd_contribs[:, start: stop], y_min=-1, y_max=25)
    fig, ax = plasmidtools.contribplots.apply_element_annotations(fig, ax, slice_start=start, slice_end=stop, flank_size=flank_size - PUFFIN_FLANK, element_size=element_size, element_label=f"{element_type}-{element_name}")
    ax.set_ylabel("Puffin (FANTOM_CAGE_fwd)", fontsize=25)
    plt.show()
